I would need to analyze a course_quiz.csv

Import the libraries

In [12]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.stats import chi2_contingency

In [2]:
#Load and look at the data

df = pd.read_csv('course_quiz.csv')

def title(text):
    display(Markdown(f'### {text}'))

#let's get basic info about it
title('Overview')
display(df)        # first and last 5 rows - get a feel for the data

title('Shape')
display(df.shape)         # (rows, columns)

title('Statistics')
display(df.describe())    # stats for numeric columns (mean, min, max...)

title('Info')
df.info()                 # column types + non-null counts

title('Nulls per column')
display(df.isnull().sum())              # null count per column


title('Duplicate Rows')
display(df.duplicated().sum())          # number of duplicate rows


title('Distinct Values per Column')
display(df.nunique())                   # distinct values per column

### Overview

,Position,Sexe,Temps,Dept,Temps_secondes,Age
0,1,M,00:32:23,76,1943,40.0
1,2,M,00:32:40,27,1960,34.0
2,3,M,00:33:15,76,1995,44.0
3,4,M,00:33:23,76,2003,46.0
4,5,M,00:33:29,76,2009,26.0
...,...,...,...,...,...,...
335,336,F,01:08:34,27,4114,41.0
336,337,F,01:08:36,27,4116,48.0
337,338,M,01:09:31,27,4171,24.0
338,339,F,01:09:31,27,4171,25.0


### Shape

(340, 6)

### Statistics

...

Position          0
Sexe              2
Temps             0
Dept              0
Temps_secondes    0
Age               4
dtype: int64

### Duplicate Rows

np.int64(0)

### Distinct Values per Column

Position          340
Sexe                3
Temps             300
Dept                4
Temps_secondes    300
Age                56
dtype: int64

1. Let's find out all the departments that took part in this run

In [7]:
df['Dept'].value_counts()

Dept
76    209
27    103
14     23
78      5
Name: count, dtype: int64

In [12]:
# Include nulls in the count
display(df['Dept'].value_counts(dropna=False))

# As percentages instead of counts
df['Dept'].value_counts(normalize=True).map('{:.1%}'.format)

Dept
76    209
27    103
14     23
78      5
Name: count, dtype: int64

Dept
76    61.5%
27    30.3%
14     6.8%
78     1.5%
Name: proportion, dtype: str

2. How many nulls ? How many variables with nulls?

In [ ]:
display(df.isnull().sum().sum()) #number of nulls

(df.isnull().sum() > 0).sum() #number of variables with nulls

np.int64(6)

np.int64(2)

3. Convert Temps to correct format. And find why it makes one of the lines null

In [23]:
df['Temps'] = pd.to_datetime(df['Temps'],format='%H:%M:%S', errors = 'coerce').dt.time

#check for nulls now
df[df['Temps'].isnull()]

,Position,Sexe,Temps,Dept,Temps_secondes,Age
250,251,M,NaT,27,0,42.0


In [27]:
#I'll reimport the csv and chec that line's 
#original Temps to find out what's the problem

df.at[250, 'Temps']

'54min47s'

It's an irregularity (a formatting inconsistency)

4. Check specific columns for outliers

(Position, Age et Temps_secondes)

In [31]:
#negative position or bigger than the number of runners
df[~df['Position'].between(1, len(df))]

# another way to get this info
# df[(df['Position'] > len(df)) | (df['Position'] < 1)]

,Position,Sexe,Temps,Dept,Temps_secondes,Age
43,-100,M,00:39:39,27,2379,31.0
264,2640,M,00:56:21,76,3381,51.0


In [32]:
#temps_secondes is less than the one for first position
#or bigger than the last position
df[~df['Temps_secondes'].between(*df['Temps_secondes'].iloc[[0, -1]])]

,Position,Sexe,Temps,Dept,Temps_secondes,Age
250,251,M,54min47s,27,0,42.0


In [35]:
#age is bigger than 100 or less than 10 but isn't null
df[~df['Age'].between(10, 100) & df['Age'].notnull()]    

,Position,Sexe,Temps,Dept,Temps_secondes,Age
302,303,M,01:00:04,76,3604,174.0


So 4 outliers

5. Check for duplicate values

In [45]:
#check for duplicates
display(df[df.duplicated(keep=False)])

#check for duplicates in specific columns subset with Age
display(df[df.duplicated(subset=['Position', 'Age'], keep=False)])
display(df[df.duplicated(subset=['Temps_secondes', 'Age'], keep=False)])
display(df[df.duplicated(subset=['Temps', 'Age'], keep=False)])

#duplicates all columns except for Position
df[df.duplicated(subset=df.columns.difference(['Position']), keep=False)]

,Position,Sexe,Temps,Dept,Temps_secondes,Age


,Position,Sexe,Temps,Dept,Temps_secondes,Age


,Position,Sexe,Temps,Dept,Temps_secondes,Age
256,257,F,00:55:35,76,3335,49.0
257,258,F,00:55:35,76,3335,49.0


,Position,Sexe,Temps,Dept,Temps_secondes,Age
256,257,F,00:55:35,76,3335,49.0
257,258,F,00:55:35,76,3335,49.0


,Position,Sexe,Temps,Dept,Temps_secondes,Age
256,257,F,00:55:35,76,3335,49.0
257,258,F,00:55:35,76,3335,49.0


Therefore there's one duplicate.

But it the quizz it's a no.

Probably because they ask if the position is the same,
and it isn't.

6. Let's see if there's still an error that I haven't seen yet

In [46]:
df['Sexe'].value_counts()

Sexe
M    244
F     92
O      2
Name: count, dtype: int64

Looks like that's an error

# Now for the last part of the class. Part 4. It's using the same csv so I'll keep doing it here

### 1. Correlation between age and position?

In [17]:
#correlation matrix of Age and Position (Pearson coefficient)
r = df['Position'].corr(df['Age'])
display(f"Pearson r = {r:.3f}")
#the closer it is to 1 or -1, the stronger the correlation. 
#Here, it's close to 0, so there's no strong linear relationship between Age and Position.

#let's see if it's different in a clean dataframe
# Keep only valid positions (between 1 and number of runners)
df_clean = df[df['Position'].between(1, len(df))]

# Keep only valid ages (between 10 and 100, or null)
df_clean = df_clean[df_clean['Age'].between(10, 100) | df_clean['Age'].isnull()]

# Drop remaining nulls and duplicates
df_clean = df_clean.drop_duplicates().dropna()

r_clean = df_clean['Position'].corr(df_clean['Age'])
display(f"Clean Pearson r = {r_clean:.3f}")

'Pearson r = 0.098'

'Clean Pearson r = 0.084'

In [18]:
df_clean.corr(numeric_only=True)

,Position,Dept,Temps_secondes,Age
Position,1.000000,-0.193608,0.927957,0.083611
Dept,-0.193608,1.000000,-0.166839,-0.035277
Temps_secondes,0.927957,-0.166839,1.000000,0.070839
Age,0.083611,-0.035277,0.070839,1.000000


In [19]:
#correlation matrix for all numeric columns
df.corr(numeric_only=True)

,Position,Dept,Temps_secondes,Age
Position,1.000000,-0.068858,0.585060,0.098384
Dept,-0.068858,1.000000,-0.152937,-0.013207
Temps_secondes,0.585060,-0.152937,1.000000,0.102820
Age,0.098384,-0.013207,0.102820,1.000000


Hence there's no correlation between position and age

### 2. Binning Age and checking correlation between Age and Position now

In [20]:
# Discretize age
df_clean['Age_bin'] = pd.cut(df_clean['Age'], bins=range(10, 110, 10))
bin_labels = [str(b) for b in df_clean['Age_bin'].cat.categories]

# Pearson coefficients
r_continuous = df_clean['Position'].corr(df_clean['Age'])
r_discrete = df_clean['Position'].corr(df_clean['Age_bin'].cat.codes)

# Mean position per bin
mean_by_bin = df_clean.groupby('Age_bin', observed=True)['Position'].mean()

# --- Build dashboard ---
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Age vs Position (scatter)',
        'Position distribution by age group (boxplot)',
        'Mean position by age group',
        ''
    ],
    specs=[[{'colspan': 2}, None], [{}, {}]],
)

# 1. Scatter (full width, top)
fig.add_trace(go.Scatter(
    x=df_clean['Age'], y=df_clean['Position'],
    mode='markers',
    marker=dict(color='mediumpurple', opacity=0.4, size=5),
    name='Runners'
), row=1, col=1)

# 2. Boxplot (bottom left)
for bin_cat in df_clean['Age_bin'].cat.categories:
    subset = df_clean[df_clean['Age_bin'] == bin_cat]['Position']
    fig.add_trace(go.Box(
        y=subset, name=str(bin_cat),
        marker_color='mediumpurple', line_color='mediumpurple',
        fillcolor='rgba(127,119,221,0.2)',
        showlegend=False
    ), row=2, col=1)

# 3. Bar — mean position (bottom right)
fig.add_trace(go.Bar(
    x=[str(b) for b in mean_by_bin.index],
    y=mean_by_bin.values,
    marker_color='mediumpurple',
    opacity=0.75,
    showlegend=False
), row=2, col=2)

# --- Layout ---
fig.update_layout(
    title=f'Age vs Position — Pearson r (continuous): {r_continuous:.3f} | r (discretized): {r_discrete:.3f}',
    height=700,
    template='plotly_white',
    showlegend=False
)

fig.update_xaxes(title_text='Age', row=1, col=1)
fig.update_yaxes(title_text='Position', row=1, col=1)
fig.update_xaxes(title_text='Age group', row=2, col=1)
fig.update_yaxes(title_text='Position', row=2, col=1)
fig.update_xaxes(title_text='Age group', row=2, col=2)
fig.update_yaxes(title_text='Mean position', row=2, col=2)

fig.show()

No correlation even binned

### 3. Linear regression

In [21]:
#clean
slope, intercept, r_value, p_value, std_err = stats.linregress(df_clean['Age'], df_clean['Position'])

print(f"a (slope)     = {slope:.4f}")
print(f"b (intercept) = {intercept:.4f}")
print(f"R²            = {r_value**2:.4f}  ({r_value**2 * 100:.2f}%)")

a (slope)     = 0.6901
b (intercept) = 141.4230
R²            = 0.0070  (0.70%)


In [22]:
#dirty
temp = df[['Age', 'Position']].dropna()

slope, intercept, r_value, p_value, std_err = stats.linregress(temp['Age'], temp['Position'])

print(f"a (slope)     = {slope:.4f}")
print(f"b (intercept) = {intercept:.4f}")
print(f"R²            = {r_value**2:.4f}  ({r_value**2 * 100:.2f}%)")

a (slope)     = 1.1797
b (intercept) = 127.5723
R²            = 0.0097  (0.97%)


### 4. Sex and Department are categorical, hence Chi-square test

In [23]:
# Build contingency table
contingency_table = pd.crosstab(df_clean['Sexe'], df_clean['Dept'])
print(contingency_table)

# Chi-square test
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print(f"\nChi² = {chi2:.4f}")
print(f"p-value = {p_value:.4f}")
print(f"Degrees of freedom = {dof}")

Dept  14  27   76  78
Sexe                 
F      5  31   55   1
M     17  68  148   4
O      0   0    2   0

Chi² = 2.3271
p-value = 0.8873
Degrees of freedom = 6


### 5. Sexe is categorical and Temps_secondes is continuous, so ANOVA

In [24]:
# Split Temps_secondes by gender group
groupes = [group['Temps_secondes'].values 
           for _, group in df_clean.groupby('Sexe')]

# ANOVA
f_stat, p_value = stats.f_oneway(*groupes)

print(f"F = {f_stat:.4f}")
print(f"p-value = {p_value:.4f}")

# Eta squared (η²) = variance explained by the group
# η² = SS_between / SS_total
grand_mean = df_clean['Temps_secondes'].mean()

ss_between = sum(
    len(group) * (group['Temps_secondes'].mean() - grand_mean)**2
    for _, group in df_clean.groupby('Sexe')
)
ss_total = ((df_clean['Temps_secondes'] - grand_mean)**2).sum()

eta2 = ss_between / ss_total

print(f"\nη² = {eta2:.4f}")

F = 41.7575
p-value = 0.0000

η² = 0.2029


light correlation